In [1]:
# DO NOT CONTAINERISE
# =====
# Dependency
# -----

# base settings
# -----
conf_vlab_name     <- "BlueCarbon"
conf_vlab_version  <- "v2"

# param_workflow_name <- "workflow"

# dev
# -----

# local
# -----

# MINIO
# -----
conf_minio_public_bucket      <- "naa-vre-public"
conf_minio_public_local_root  <- "/home/jovyan/Cloud Storage/naa-vre-public/vl-bluecarbon/v2"
conf_minio_public_local_code  <- "/home/jovyan/Cloud Storage/naa-vre-public/vl-bluecarbon/v2/code"
conf_minio_public_local_data  <- "/home/jovyan/Cloud Storage/naa-vre-public/vl-bluecarbon/v2/data"

conf_minio_user_bucket        <- "naa-vre-user-data"
conf_minio_user_local_root    <- "/home/jovyan/Cloud Storage/naa-vre-user-data/BlueCarbon"
conf_minio_user_local_code    <- "/home/jovyan/Cloud Storage/naa-vre-user-data/BlueCarbon/library"
conf_minio_user_local_data    <- "/home/jovyan/Cloud Storage/naa-vre-user-data/BlueCarbon"         # param_workflow_name
conf_minio_user_local_flog    <- "/home/jovyan/Cloud Storage/naa-vre-user-data/BlueCarbon/log.md"  # param_workflow_name

# API key
# -----

# Input param
# -----
conf_delimiter_tsv <- "\t"
conf_delimiter_csv <- ","

# GUI, File
# .....
# # param_file <- "Seagrass_site_data.csv"
# param_file <- "Seagrass_site_data.xlsx"

param_latitude <- 56.0953
param_longitude <- 14.2785
param_seagrass_species <- "Posidonia oceanica"

# Output param
# -----
# results
conf_fname_tmp_result <- "tmp_result.rds"
conf_fname_result_txt <- "Seagrass_site_result.txt"
conf_fname_result_csv <- "Seagrass_site_result.csv"

cat("Finish: NaaVRE parameters", sep="\n")
cat("Workspace public:", sep="\n")
print(paste("  Root", conf_minio_public_local_root, sep=": "))
print(paste("  Code", conf_minio_public_local_code, sep=": "))
print(paste("  Data", conf_minio_public_local_data, sep=": "))

cat("Workspace user:", sep="\n")
print(paste("  Root", conf_minio_user_local_root, sep=": "))
print(paste("  Code", conf_minio_user_local_code, sep=": "))
print(paste("  Data", conf_minio_user_local_data, sep=": "))
print(paste("  Log",  conf_minio_user_local_flog, sep=":  "))


Finish: NaaVRE parameters
Workspace public:
[1] "  Root: /home/jovyan/Cloud Storage/naa-vre-public/vl-bluecarbon/v2"
[1] "  Code: /home/jovyan/Cloud Storage/naa-vre-public/vl-bluecarbon/v2/code"
[1] "  Data: /home/jovyan/Cloud Storage/naa-vre-public/vl-bluecarbon/v2/data"
Workspace user:
[1] "  Root: /home/jovyan/Cloud Storage/naa-vre-user-data/BlueCarbon"
[1] "  Code: /home/jovyan/Cloud Storage/naa-vre-user-data/BlueCarbon/library"
[1] "  Data: /home/jovyan/Cloud Storage/naa-vre-user-data/BlueCarbon"
[1] "  Log:  /home/jovyan/Cloud Storage/naa-vre-user-data/BlueCarbon/log.md"


In [2]:
# BlueCarbon, GUI, 1-Input
library(dplyr)

library(readxl)
library(readr)
library(RNetCDF)


# prepare folders
# -----

# GUI
# -----
seagrass_species_options <- list(
    "Cymodocea nodosa", 
    "Halophila stipulacea", 
    "Posidonia oceanica", 
    "Zostera marina", 
    "Zostera noltei", 
    "Zostera marina and Cymodocea nodosa", 
    "Zostera marina and Zostera noltei",
    "Unspecified"
)

validate_seagrass_species <- function(tmp_seagrass_species, options) {
    is_valid_option <- tmp_seagrass_species %in% options
    
    if (tmp_seagrass_species == "") {
        message("Seagrass species is unspecified")
    } else if (!is_valid_option) {
        stop(
            tmp_seagrass_species,
            "Is not part of the options of allowed seagrass species: ", 
            paste(options, collapse = ", ")
        )
    } else {
        message(tmp_seagrass_species, "is a valid the seagrass species.")
    }
}

validate_geographic_coordinate <- function(coordinate_double) {  
    if (!typeof(coordinate_double) == "double") {
        stop(
            "The coordinate: ", coordinate_double, " is not a double (e.g. 56.0953, not `56.0953`)"
        )
    }
    # coordinate_string <- as.character(coordinate_double)
    # pattern <- "^\d{2}\.\d{4}$"
    # is_valid_format <- grepl(pattern, coordinate_string)
    # if (!(is_valid_format)) {
    #     stop(
    #         "The coordinate: ", coordinate_double, " does not follow the required format: ", pattern, " e.g. 56.0953"
    #     )
    # } else {
    #     message(coordinate_double, " is valid geographic coordinate input.")
    # }
}

validate_seagrass_species(param_seagrass_species, seagrass_species_options)
validate_geographic_coordinate(param_latitude)
validate_geographic_coordinate(param_longitude)

# input
# -----
tmp_str <- paste("seagrass_species", param_seagrass_species, "; ", "latitude", param_latitude, "; ", "longitude", param_longitude, "\n")
print(tmp_str)
if (file.exists(conf_minio_user_local_flog)) {
  cat(tmp_str, file=conf_minio_user_local_flog, sep="\n", append=TRUE)
} else {
  cat(tmp_str, file=conf_minio_user_local_flog, sep="\n")
}

RASTER_BASE <- file.path(conf_minio_public_local_data)
MODEL_PATH  <- file.path(conf_minio_public_local_data, "GPR_final.rds")

# output
# -----
str_time <- format(Sys.time(),  "%Y%m%d_%H%M%S")

fname_tmp_result <- paste(str_time, conf_fname_tmp_result, sep ="-")
file_tmp_result <- file.path(conf_minio_user_local_data, fname_tmp_result)

fname_result_csv <- paste(str_time, conf_fname_result_csv, sep ="-")
fname_result_txt <- paste(str_time, conf_fname_result_txt, sep ="-")

# func
# -----
# Parameter validator, GUI
# =====

# start
# -----
# .....
# User configuration
# .....
POINT_COLUMNS <- c("longitude", "latitude", "seagrass_species")
USE_CLOSEST_RASTER_VALUE <- TRUE
RASTER_METHOD <- "nearest" # "nearest" or "bilinear"
RETURN_PREDICTION_SE <- TRUE
PREDICTION_MEAN_COLUMN <- "predicted_carbon_density"
PREDICTION_SE_COLUMN <- "predicted_se"
RS_COVARIATES <- c(
  "S3A_OLCI_ERRNT.20160425_20241231.L3m.CU.CHL.chlor_a.4km",
  "S3A_OLCI_ERRNT.20160425_20241231.L3m.CU.IOP.adg_443.4km",
  "S3A_OLCI_ERRNT.20160425_20241231.L3m.CU.IOP.bbp_443.4km",
  "S3A_OLCI_ERRNT.20160425_20241231.L3m.CU.KD.Kd_490.4km",
  "S3A_OLCI_ERRNT.20160425_20241231.L3m.CU.RRS.Rrs_443.4km",
  "S3A_OLCI_ERRNT.20160425_20241231.L3m.CU.RRS.Rrs_620.4km"
)


# .....
# Load input points
# .....
input_data <- data.frame(
    latitude=as.vector(as.numeric(param_latitude)),
    longitude=as.vector(as.numeric(param_longitude)),
    seagrass_species=as.vector(unlist(param_seagrass_species)))
input_data

# Load the saved GPR model.
model <- readRDS(MODEL_PATH)
if (!identical(model$model_type, "GPR") && !inherits(model$model, "GauPro")) {
  stop("The model file does not contain a GPR model: ", MODEL_PATH, call. = FALSE)
}
PREDICTOR_VARS <- model$predictor_vars
if (is.null(PREDICTOR_VARS) || length(PREDICTOR_VARS) == 0L) {
  stop("The GPR model does not contain predictor_vars.", call. = FALSE)
}
MISSING_COLUMNS <- setdiff(POINT_COLUMNS, names(input_data))
if (length(MISSING_COLUMNS) > 0L) {
  stop(
    "Input data is missing required column(s): ",
    paste(MISSING_COLUMNS, collapse = ", "),
    call. = FALSE
  )
}
POINT_DATA_COLUMNS <- unique(c(
  POINT_COLUMNS,
  intersect(PREDICTOR_VARS, names(input_data))
))
POINT_DATA <- as.data.frame(input_data[, POINT_DATA_COLUMNS, drop = FALSE])
tmp_str <- paste("POINT_DATA:", head(POINT_DATA), "\n")
print(tmp_str)
cat(tmp_str, file=conf_minio_user_local_flog, sep="\n", append=TRUE)

RASTER_COVARIATES <- setdiff(PREDICTOR_VARS, names(POINT_DATA))

tmp_str <- paste("Input points:", nrow(input_data), " | Model predictors:", length(PREDICTOR_VARS), "\n")
print(tmp_str)
cat(tmp_str, file=conf_minio_user_local_flog, sep="\n", append=TRUE)

tmp_str <- paste("Raster covariates to extract:", length(RASTER_COVARIATES), "\n")
print(tmp_str)
cat(tmp_str, file=conf_minio_user_local_flog, sep="\n", append=TRUE)
if (length(RASTER_COVARIATES) > 0L) {
  RASTER_PACKAGES <- c("RNetCDF", "terra")
  MISSING_PACKAGES <- RASTER_PACKAGES[
    !vapply(RASTER_PACKAGES, requireNamespace, logical(1), quietly = TRUE)
  ]
  if (length(MISSING_PACKAGES) > 0L) {
    stop(
      "Missing required package(s) for raster extraction: ",
      paste(MISSING_PACKAGES, collapse = ", "),
      ". Restore the v2 renv environment or install these packages.",
      call. = FALSE
    )
  }
}

# .....
# Match each required predictor to a NetCDF file and its 2D latitude/longitude variable
# .....
RASTER_CONFIG <- list()
REQUESTED_COVARIATES <- unique(tolower(RASTER_COVARIATES))
if (length(REQUESTED_COVARIATES) > 0L) {
  if (!dir.exists(RASTER_BASE)) stop("Raster directory not found: ", RASTER_BASE, call. = FALSE)
  RASTER_FILES <- list.files(
    RASTER_BASE, pattern = "[.]nc$", full.names = TRUE, ignore.case = TRUE
  )
  RASTER_FILE_NAMES <- tolower(gsub(
    "-", "_", sub("[.]nc$", "", basename(RASTER_FILES), ignore.case = TRUE), fixed = TRUE
  ))
  if (anyDuplicated(RASTER_FILE_NAMES)) {
    stop("Raster filenames are ambiguous after normalization in: ", RASTER_BASE, call. = FALSE)
  }
  MISSING_RASTERS <- setdiff(REQUESTED_COVARIATES, RASTER_FILE_NAMES)
  if (length(MISSING_RASTERS) > 0L) {
    stop(
      "No NetCDF raster found for required covariate(s): ",
      paste(MISSING_RASTERS, collapse = ", "),
      call. = FALSE
    )
  }

  # Loop through each requested covariate and find its corresponding NetCDF file and 2D lat/lon variable
  for (COVARIATE in REQUESTED_COVARIATES) {
    RASTER_FILE <- RASTER_FILES[match(COVARIATE, RASTER_FILE_NAMES)]

    # Open the NetCDF file
    NC_HANDLE <- RNetCDF::open.nc(RASTER_FILE)
    NC_VARIABLE_NAME <- NULL
    NC_VARIABLE_COUNT <- RNetCDF::file.inq.nc(NC_HANDLE)$nvars
    for (NC_INDEX in seq_len(NC_VARIABLE_COUNT) - 1L) {
      NC_VARIABLE <- RNetCDF::var.inq.nc(NC_HANDLE, NC_INDEX)
      if (length(NC_VARIABLE$dimids) != 2L) next
      
      NC_DIMENSION_NAMES <- character()
      for (NC_DIMENSION_ID in NC_VARIABLE$dimids) {
        NC_DIMENSION_NAMES <- c(
          NC_DIMENSION_NAMES,
          RNetCDF::dim.inq.nc(NC_HANDLE, NC_DIMENSION_ID)$name
        )
      }
      if (any(grepl("lat", NC_DIMENSION_NAMES, ignore.case = TRUE)) &&
          any(grepl("lon", NC_DIMENSION_NAMES, ignore.case = TRUE))) {
        NC_VARIABLE_NAME <- NC_VARIABLE$name
        break
      }
    }
    RNetCDF::close.nc(NC_HANDLE)

    # Store the raster configuration for the current covariate
    if (is.null(NC_VARIABLE_NAME)) {
      stop(
        "No 2D latitude/longitude variable found in raster: ", basename(RASTER_FILE),
        call. = FALSE
      )
    }
    RASTER_CONFIG[[COVARIATE]] <- list(
      file = file.path(RASTER_BASE, basename(RASTER_FILE)),
      varname = NC_VARIABLE_NAME
    )
  }
}
# Return the final raster configuration for all requested covariates
tmp_str <- paste("RASTER_CONFIG", RASTER_CONFIG, "\n")
print(tmp_str)
cat(tmp_str, file=conf_minio_user_local_flog, sep="\n", append=TRUE)

# .....
# Extract environmental covariates at each point
# .....
PREDICTION_DATA <- POINT_DATA
if (length(RASTER_COVARIATES) > 0L) {
  POINT_COORDINATES <- as.matrix(POINT_DATA[, c("longitude", "latitude"), drop = FALSE])
  for (COVARIATE in RASTER_COVARIATES) {
    COVARIATE_CONFIG <- RASTER_CONFIG[[tolower(COVARIATE)]]
    if (is.null(COVARIATE_CONFIG)) {
      stop("Raster configuration not found for covariate: ", COVARIATE, call. = FALSE)
    }

    # Construct the full path to the NetCDF raster file
    RASTER_PATH <- COVARIATE_CONFIG$file
    if (!file.exists(RASTER_PATH)) stop("NetCDF file not found: ", RASTER_PATH, call. = FALSE)

    RASTER <- terra::rast(RASTER_PATH)
    if (!is.null(COVARIATE_CONFIG$varname) &&
        nzchar(COVARIATE_CONFIG$varname) &&
        COVARIATE_CONFIG$varname %in% names(RASTER)) {
      RASTER <- RASTER[[COVARIATE_CONFIG$varname]]
    } else if (terra::nlyr(RASTER) > 1L) {
      warning(
        "Variable '", COVARIATE_CONFIG$varname, "' not found in ",
        COVARIATE_CONFIG$file, "; using first layer in file. Check the raster configuration."
      )
      RASTER <- RASTER[[1L]]
    }

    # Extract the raster values at the point locations
    TERRA_METHOD <- if (identical(tolower(RASTER_METHOD), "bilinear")) "bilinear" else "simple"
    COVARIATE_VALUES <- as.numeric(
      terra::extract(RASTER, POINT_COORDINATES, method = TERRA_METHOD)[, 1L]
    )
    if (USE_CLOSEST_RASTER_VALUE && anyNA(COVARIATE_VALUES) &&
        requireNamespace("FNN", quietly = TRUE)) {
      MISSING_POINT_INDICES <- which(is.na(COVARIATE_VALUES))
      RASTER_VALUES <- terra::values(RASTER, mat = FALSE)
      VALID_CELL_INDICES <- which(!is.na(RASTER_VALUES))
      
      # Find the nearest valid raster value for each missing point
      if (length(VALID_CELL_INDICES) > 0L) {
        NEAREST_VALID_INDICES <- FNN::get.knnx(
          data = terra::xyFromCell(RASTER, VALID_CELL_INDICES),
          query = POINT_COORDINATES[MISSING_POINT_INDICES, , drop = FALSE],
          k = 1L
        )$nn.index[, 1L]
        COVARIATE_VALUES[MISSING_POINT_INDICES] <-
          RASTER_VALUES[VALID_CELL_INDICES][NEAREST_VALID_INDICES]
      }
    }
    PREDICTION_DATA[[COVARIATE]] <- COVARIATE_VALUES
  }
}

# Match the non-negative preprocessing used for remote-sensing covariates.
EXISTING_RS_COVARIATES <- intersect(RS_COVARIATES, colnames(PREDICTION_DATA))
for (COVARIATE in EXISTING_RS_COVARIATES) {
  if (is.numeric(PREDICTION_DATA[[COVARIATE]])) {
    PREDICTION_DATA[[COVARIATE]] <- pmax(PREDICTION_DATA[[COVARIATE]], 0, na.rm = FALSE)
  }
}
# Identify any raster covariates for which extraction failed (i.e., contains NA values)
MISSING_EXTRACTED_COVARIATES <- character()
for (COVARIATE in RASTER_COVARIATES) {
  if (any(is.na(PREDICTION_DATA[[COVARIATE]]))) {
    MISSING_EXTRACTED_COVARIATES <- c(MISSING_EXTRACTED_COVARIATES, COVARIATE)
  }
}
if (length(MISSING_EXTRACTED_COVARIATES) > 0L) {
  warning(
    "Missing extracted covariate values: ",
    paste(MISSING_EXTRACTED_COVARIATES, collapse = ", "),
    call. = FALSE
  )
}
# Prepared point covariates
tmp_str <- paste("Prepared point covariates:", nrow(PREDICTION_DATA), "rows x", ncol(PREDICTION_DATA), "columns\n", "\n")
print(tmp_str)
cat(tmp_str, file=conf_minio_user_local_flog, sep="\n", append=TRUE)

# .....
# Apply the saved model's categorical encoding and scaling
# .....
MISSING_PREDICTORS <- setdiff(PREDICTOR_VARS, names(PREDICTION_DATA))
if (length(MISSING_PREDICTORS) > 0L) {
  stop(
    "Prediction data is missing model predictor(s): ",
    paste(MISSING_PREDICTORS, collapse = ", "),
    call. = FALSE
  )
}
MODEL_COLUMNS <- if (is.null(model$encoded_names)) PREDICTOR_VARS else model$encoded_names
ENCODING <- model$encoding
ENCODING_TYPE <- if (is.null(ENCODING) || is.null(ENCODING$type)) NA_character_ else ENCODING$type
if (identical(ENCODING_TYPE, "one_hot") &&
    !is.null(model$species_means) &&
    !is.null(ENCODING$kernel_predictor_vars)) {
  ENCODING_TYPE <- "gpr_species_adjusted"
}
if (identical(ENCODING_TYPE, "gpr_species_adjusted")) {
  KERNEL_PREDICTORS <- if (is.null(ENCODING$kernel_predictor_vars)) {
    setdiff(PREDICTOR_VARS, "seagrass_species")
  } else {
    ENCODING$kernel_predictor_vars
  }
  X <- as.data.frame(PREDICTION_DATA[, KERNEL_PREDICTORS, drop = FALSE])
  ONE_HOT_PREDICTORS <- if (is.null(ENCODING$predictor_vars)) names(X) else ENCODING$predictor_vars
  X <- as.data.frame(X[, ONE_HOT_PREDICTORS, drop = FALSE])
  if (!is.null(ENCODING$levels) && length(ENCODING$levels) > 0L) {
    for (VARIABLE in names(ENCODING$levels)) {
      if (VARIABLE %in% names(X)) {
        X[[VARIABLE]] <- factor(
          as.character(X[[VARIABLE]]),
          levels = ENCODING$levels[[VARIABLE]]
        )
      }
    }
  }
  # Convert categorical variables to one-hot encoding
  X <- as.data.frame(model.matrix(~ . - 1, data = X))
  MISSING_MODEL_COLUMNS <- setdiff(MODEL_COLUMNS, names(X))
  for (VARIABLE in MISSING_MODEL_COLUMNS) X[[VARIABLE]] <- 0
  X <- X[, MODEL_COLUMNS, drop = FALSE]
} else if (identical(ENCODING_TYPE, "one_hot")) {
  X <- as.data.frame(PREDICTION_DATA[, PREDICTOR_VARS, drop = FALSE])
  ONE_HOT_PREDICTORS <- if (is.null(ENCODING$predictor_vars)) names(X) else ENCODING$predictor_vars

  # Subset the prediction data to only include the one-hot predictors
  X <- as.data.frame(X[, ONE_HOT_PREDICTORS, drop = FALSE])
  if (!is.null(ENCODING$levels) && length(ENCODING$levels) > 0L) {
    for (VARIABLE in names(ENCODING$levels)) {
      if (VARIABLE %in% names(X)) {
        X[[VARIABLE]] <- factor(
          as.character(X[[VARIABLE]]),
          levels = ENCODING$levels[[VARIABLE]]
        )
      }
    }
  }

  # Convert categorical variables to one-hot encoding
  X <- as.data.frame(model.matrix(~ . - 1, data = X))
  MISSING_MODEL_COLUMNS <- setdiff(MODEL_COLUMNS, names(X))
  for (VARIABLE in MISSING_MODEL_COLUMNS) X[[VARIABLE]] <- 0

  # Ensure the columns in the prediction data match the model columns
  X <- X[, MODEL_COLUMNS, drop = FALSE]
} else {
  X <- as.data.frame(PREDICTION_DATA[, PREDICTOR_VARS, drop = FALSE])
  if (!is.null(ENCODING) && !is.null(ENCODING$levels) && length(ENCODING$levels) > 0L) {
    for (VARIABLE in names(ENCODING$levels)) {
      if (VARIABLE %in% names(X)) {
        X[[VARIABLE]] <- as.integer(factor(
          as.character(X[[VARIABLE]]),
          levels = ENCODING$levels[[VARIABLE]]
        ))
        X[[VARIABLE]][is.na(X[[VARIABLE]])] <- 1L
      }
    }
  }
}

# Scale the prediction data using the model's scale parameters
SCALE_PARAMS <- model$scale_params
if (!is.null(SCALE_PARAMS) && is.null(SCALE_PARAMS$means) && !is.null(SCALE_PARAMS$x_center)) {
  SCALE_PARAMS <- list(means = SCALE_PARAMS$x_center, sds = SCALE_PARAMS$x_scale)
}
if (!is.null(SCALE_PARAMS) && length(SCALE_PARAMS$means) > 0L) {
  for (VARIABLE in names(SCALE_PARAMS$means)) {
    if (VARIABLE %in% names(X)) {
      X[[VARIABLE]] <- (X[[VARIABLE]] - SCALE_PARAMS$means[VARIABLE]) /
        SCALE_PARAMS$sds[VARIABLE]
    }
  }
}
print("X")
head(X)

# Convert the prediction data to a matrix suitable for GPR model input
X_MATRIX <- as.matrix(X[, MODEL_COLUMNS, drop = FALSE])
storage.mode(X_MATRIX) <- "double"
dim(X_MATRIX)

# .....
# Predict carbon density and optional standard error
# .....
tmp_str <- paste("Predicting carbon density for", nrow(input_data), "point(s)...\n")
print(tmp_str)
cat(tmp_str, file=conf_minio_user_local_flog, sep="\n", append=TRUE)

PREDICTION_OUTPUT <- model$model$pred(
  X_MATRIX,
  se.fit = RETURN_PREDICTION_SE,
  return_df = TRUE
)
if (is.list(PREDICTION_OUTPUT) && !is.null(PREDICTION_OUTPUT$mean)) {
  PREDICTION_MEAN <- as.numeric(PREDICTION_OUTPUT$mean)
  PREDICTION_SE <- if (RETURN_PREDICTION_SE) as.numeric(PREDICTION_OUTPUT$se) else NULL
} else {
  PREDICTION_MEAN <- as.numeric(PREDICTION_OUTPUT)
  PREDICTION_SE <- NULL
}

# Apply species-specific adjustment to the predicted carbon density if available
SPECIES_VARIABLE <- if (is.null(ENCODING) || is.null(ENCODING$species_var)) {
  NULL
} else {
  ENCODING$species_var
}
if (!is.null(model$species_means) &&
    !is.null(SPECIES_VARIABLE) &&
    SPECIES_VARIABLE %in% names(PREDICTION_DATA)) {
  SPECIES_FALLBACK <- if (is.null(model$global_mean)) 0 else model$global_mean
  SPECIES_ADJUSTMENT <- as.numeric(
    model$species_means[as.character(PREDICTION_DATA[[SPECIES_VARIABLE]])]
  )
  SPECIES_ADJUSTMENT[is.na(SPECIES_ADJUSTMENT)] <- SPECIES_FALLBACK
  PREDICTION_MEAN <- PREDICTION_MEAN + SPECIES_ADJUSTMENT
}

# Apply log transformation if the model was trained on log-transformed response
if (isTRUE(model$log_response)) {
  PREDICTION_MEAN <- exp(PREDICTION_MEAN)
  if (!is.null(PREDICTION_SE)) PREDICTION_SE <- PREDICTION_SE * PREDICTION_MEAN
}
if (length(PREDICTION_MEAN) != nrow(input_data)) {
  stop("GPR prediction count does not match the number of input rows.", call. = FALSE)
}

# Check that the number of predictions matches the number of input rows
SG_modeling_dataframe <- PREDICTION_DATA

SG_modeling_dataframe[[PREDICTION_MEAN_COLUMN]] <- PREDICTION_MEAN
if (!is.null(PREDICTION_SE)) SG_modeling_dataframe[[PREDICTION_SE_COLUMN]] <- PREDICTION_SE

head(SG_modeling_dataframe)

# save dataframe, SG_modeling_dataframe
saveRDS(SG_modeling_dataframe, file_tmp_result)
# write.csv(SG_modeling_dataframe, file_tmp_result)

# -----
# cat("RASTER_BASE:", RASTER_BASE, "\n")
# cat("MODEL_PATH:",  MODEL_PATH, "\n")
if (!file.exists(MODEL_PATH)) stop("GPR model file not found: ", MODEL_PATH, call. = FALSE)
if (!dir.exists(RASTER_BASE)) stop("Raster directory not found: ", RASTER_BASE, call. = FALSE)

# dataframe
tmp_str <- paste("file_tmp_result:", file_tmp_result, "\n")
print(tmp_str)
cat(tmp_str, file=conf_minio_user_local_flog, sep="\n", append=TRUE)



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Posidonia oceanicais a valid the seagrass species.



[1] "seagrass_species Posidonia oceanica ;  latitude 56.0953 ;  longitude 14.2785 \n"


latitude,longitude,seagrass_species
<dbl>,<dbl>,<chr>
56.0953,14.2785,Posidonia oceanica


[1] "POINT_DATA: 14.2785 \n"            "POINT_DATA: 56.0953 \n"           
[3] "POINT_DATA: Posidonia oceanica \n"
[1] "Input points: 1  | Model predictors: 14 \n"
[1] "Raster covariates to extract: 13 \n"
 [1] "RASTER_CONFIG list(file = \"/home/jovyan/Cloud Storage/naa-vre-public/vl-bluecarbon/v2/data/Surf_fgco2_mean_molC_m2_yr.nc\", varname = \"Surf_fgco2_mean_molC_m2_yr\") \n"          
 [2] "RASTER_CONFIG list(file = \"/home/jovyan/Cloud Storage/naa-vre-public/vl-bluecarbon/v2/data/S3A_OLCI_ERRNT.20160425_20241231.L3m.CU.IOP.adg_443.4km.nc\", varname = \"adg_443\") \n"
 [3] "RASTER_CONFIG list(file = \"/home/jovyan/Cloud Storage/naa-vre-public/vl-bluecarbon/v2/data/vo_mean_1.5m_m_s.nc\", varname = \"vo_mean_1.5m_m_s\") \n"                              
 [4] "RASTER_CONFIG list(file = \"/home/jovyan/Cloud Storage/naa-vre-public/vl-bluecarbon/v2/data/uo_p90_1.5m_m_s.nc\", varname = \"uo_p90_1.5m_m_s\") \n"                                
 [5] "RASTER_CONFIG list(file = \"/home/jovya

,surf_fgco2_mean_molc_m2_yr,s3a_olci_errnt.20160425_20241231.l3m.cu.iop.adg_443.4km,vo_mean_1.5m_m_s,uo_p90_1.5m_m_s,vo_p90_1.5m_m_s,sw_ke_mean_cm2_s2,daily_std_wave_height_mean_m,fe_mean_monthly_1.5m_mmol_m3,bottomt_p95_daily_c,spco2_monthly_mean_pa,s3a_olci_errnt.20160425_20241231.l3m.cu.chl.chlor_a.4km,s3a_olci_errnt.20160425_20241231.l3m.cu.rrs.rrs_620.4km,no_sig_wave_heights_mean
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,-0.4615585,-0.5072494,0.7920087,-0.008177953,0.4309543,1.403522,-0.3299884,1.330387,0.997217,0.5113913,1.749577,-0.4633334,0.1681881


[1]  1 13

[1] "Predicting carbon density for 1 point(s)...\n"


,longitude,latitude,seagrass_species,surf_fgco2_mean_molc_m2_yr,s3a_olci_errnt.20160425_20241231.l3m.cu.iop.adg_443.4km,vo_mean_1.5m_m_s,uo_p90_1.5m_m_s,vo_p90_1.5m_m_s,sw_ke_mean_cm2_s2,daily_std_wave_height_mean_m,fe_mean_monthly_1.5m_mmol_m3,bottomt_p95_daily_c,spco2_monthly_mean_pa,s3a_olci_errnt.20160425_20241231.l3m.cu.chl.chlor_a.4km,s3a_olci_errnt.20160425_20241231.l3m.cu.rrs.rrs_620.4km,no_sig_wave_heights_mean,predicted_carbon_density,predicted_se
,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,14.2785,56.0953,Posidonia oceanica,0.5273098,0.08080006,0.01808088,0.06591998,0.05493332,75.37934,0.2185655,0.004651441,18.81483,39.33704,27.08706,0.0002560009,37.84513,0.01876371,0.0167864


[1] "file_tmp_result: /home/jovyan/Cloud Storage/naa-vre-user-data/BlueCarbon/20261104_123020-tmp_result.rds \n"


In [3]:
# BlueCarbon, GUI, 2-Calculate Carbon Stock

library(dplyr)

# library(readxl)
# library(readr)
# library(RNetCDF)

# input
# -----
tmp_str <- paste("file_tmp_result:", file_tmp_result, "\n")
print(tmp_str)
if (file.exists(conf_minio_user_local_flog)) {
  cat(tmp_str, file=conf_minio_user_local_flog, sep="\n")
} else {
  cat(tmp_str, file=conf_minio_user_local_flog, sep="\n", append=TRUE)
}

tmp_str <- paste("fname_result_csv:", fname_result_csv, "\n")
print(tmp_str)
cat(tmp_str, file=conf_minio_user_local_flog, sep="\n", append=TRUE)

tmp_str <- paste("fname_result_txt:", fname_result_txt, "\n")
print(tmp_str)
cat(tmp_str, file=conf_minio_user_local_flog, sep="\n", append=TRUE)

# output
# -----
file_result_csv <- file.path(conf_minio_user_local_data, fname_result_csv)
file_result_txt <- file.path(conf_minio_user_local_data, fname_result_txt)

tmp_str <- paste("file_result_csv:", file_result_csv, "\n")
print(tmp_str)
cat(tmp_str, file=conf_minio_user_local_flog, sep="\n", append=TRUE)

tmp_str <- paste("file_result_txt:", file_result_txt, "\n")
print(tmp_str)
cat(tmp_str, file=conf_minio_user_local_flog, sep="\n", append=TRUE)

# Carbon stock in the upper 30 and 100 cm
tmp_str <- paste("Carbon stock in the upper 30 and 100 cm", "\n")
print(tmp_str)
cat(tmp_str, file=conf_minio_user_local_flog, sep="\n", append=TRUE)

SG_modeling_dataframe <- readRDS(file_tmp_result)
# SG_modeling_dataframe <- input_data

# depths <- c(5, 15, 25, 35, 45, 55, 65, 75, 85, 95)

head(SG_modeling_dataframe)

# =====
# Calculate carbon stock per 10 cm
# Carbon stock (MgC/ha)=Carbon density (gC/cm³) × Depth (cm) × 100,000,000 (cm2/ha) ÷ 1,000,000 (g/Mg)
# Where:
#   1 hectare = 10,000 m² = 100,000,000 cm²
#   1 Mg = 1,000,000 g
SG_modeling_dataframe$carbon_stock_Mg_ha_upper30cm  = SG_modeling_dataframe$predicted_carbon_density* 3*10*100000000*(1/1000000)
SG_modeling_dataframe$carbon_stock_Mg_ha_upper100cm = SG_modeling_dataframe$predicted_carbon_density*10*10*100000000*(1/1000000)
SG_modeling_dataframe$carbon_stock_Mg_ha_upper30cm_se  = SG_modeling_dataframe$predicted_se* 3*10*100000000*(1/1000000)
SG_modeling_dataframe$carbon_stock_Mg_ha_upper100cm_se = SG_modeling_dataframe$predicted_se*10*10*100000000*(1/1000000)
head(SG_modeling_dataframe)

# Create sample_ID column
SG_modeling_dataframe$sample_ID <- rep(1:(nrow(SG_modeling_dataframe)/1), each = 1)
head(SG_modeling_dataframe)

# # #####
# # Create summary sentences

summary_dataframe <- SG_modeling_dataframe
# summary_dataframe <- SG_modeling_dataframe %>% 
#   select(-c("predicted_carbon_density"
#             "predicted_se"))

summary_sentences <- SG_modeling_dataframe %>%
  mutate(summary = paste0(
    "For the seagrass bed at latitude ", latitude, " and longitude ", longitude, "\n",
    "The species identity ", seagrass_species, "\n",
    "The predicted carbon stock in the upper \n",
    "  - upper 30cm  of the sediment is ", format(carbon_stock_Mg_ha_upper30cm,  digits=3, nsmall=3), " Mg/ha\n",
    "  - upper 100cm of the sediment is ", format(carbon_stock_Mg_ha_upper100cm, digits=3, nsmall=3), " Mg/ha\n",
    "The predicted carbon stock standard error in the upper \n",
    "  - upper 30cm  of the sediment is ", format(carbon_stock_Mg_ha_upper30cm_se,  digits=3, nsmall=3), " Mg/ha\n",
    "  - upper 100cm of the sediment is ", format(carbon_stock_Mg_ha_upper100cm_se, digits=3, nsmall=3), " Mg/ha\n"
  )) %>%
  pull(summary)

# Save result
# -----
# summary_dataframe
# summary(summary_dataframe)
write.csv(summary_dataframe, file_result_csv)

# summary_sentences
# cat(summary_sentences, sep = "\n")
cat(summary_sentences, file=file_result_txt, sep="\n")

tmp_str <- paste("Carbon predicted", "\n")
print(tmp_str)
cat(tmp_str, file=conf_minio_user_local_flog, sep="\n", append=TRUE)


[1] "file_tmp_result: /home/jovyan/Cloud Storage/naa-vre-user-data/BlueCarbon/20261104_123020-tmp_result.rds \n"
[1] "fname_result_csv: 20261104_123020-Seagrass_site_result.csv \n"
[1] "fname_result_txt: 20261104_123020-Seagrass_site_result.txt \n"
[1] "file_result_csv: /home/jovyan/Cloud Storage/naa-vre-user-data/BlueCarbon/20261104_123020-Seagrass_site_result.csv \n"
[1] "file_result_txt: /home/jovyan/Cloud Storage/naa-vre-user-data/BlueCarbon/20261104_123020-Seagrass_site_result.txt \n"
[1] "Carbon stock in the upper 30 and 100 cm \n"


,longitude,latitude,seagrass_species,surf_fgco2_mean_molc_m2_yr,s3a_olci_errnt.20160425_20241231.l3m.cu.iop.adg_443.4km,vo_mean_1.5m_m_s,uo_p90_1.5m_m_s,vo_p90_1.5m_m_s,sw_ke_mean_cm2_s2,daily_std_wave_height_mean_m,fe_mean_monthly_1.5m_mmol_m3,bottomt_p95_daily_c,spco2_monthly_mean_pa,s3a_olci_errnt.20160425_20241231.l3m.cu.chl.chlor_a.4km,s3a_olci_errnt.20160425_20241231.l3m.cu.rrs.rrs_620.4km,no_sig_wave_heights_mean,predicted_carbon_density,predicted_se
,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,14.2785,56.0953,Posidonia oceanica,0.5273098,0.08080006,0.01808088,0.06591998,0.05493332,75.37934,0.2185655,0.004651441,18.81483,39.33704,27.08706,0.0002560009,37.84513,0.01876371,0.0167864


,longitude,latitude,seagrass_species,surf_fgco2_mean_molc_m2_yr,s3a_olci_errnt.20160425_20241231.l3m.cu.iop.adg_443.4km,vo_mean_1.5m_m_s,uo_p90_1.5m_m_s,vo_p90_1.5m_m_s,sw_ke_mean_cm2_s2,daily_std_wave_height_mean_m,⋯,spco2_monthly_mean_pa,s3a_olci_errnt.20160425_20241231.l3m.cu.chl.chlor_a.4km,s3a_olci_errnt.20160425_20241231.l3m.cu.rrs.rrs_620.4km,no_sig_wave_heights_mean,predicted_carbon_density,predicted_se,carbon_stock_Mg_ha_upper30cm,carbon_stock_Mg_ha_upper100cm,carbon_stock_Mg_ha_upper30cm_se,carbon_stock_Mg_ha_upper100cm_se
,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,14.2785,56.0953,Posidonia oceanica,0.5273098,0.08080006,0.01808088,0.06591998,0.05493332,75.37934,0.2185655,⋯,39.33704,27.08706,0.0002560009,37.84513,0.01876371,0.0167864,56.29113,187.6371,50.3592,167.864


,longitude,latitude,seagrass_species,surf_fgco2_mean_molc_m2_yr,s3a_olci_errnt.20160425_20241231.l3m.cu.iop.adg_443.4km,vo_mean_1.5m_m_s,uo_p90_1.5m_m_s,vo_p90_1.5m_m_s,sw_ke_mean_cm2_s2,daily_std_wave_height_mean_m,⋯,s3a_olci_errnt.20160425_20241231.l3m.cu.chl.chlor_a.4km,s3a_olci_errnt.20160425_20241231.l3m.cu.rrs.rrs_620.4km,no_sig_wave_heights_mean,predicted_carbon_density,predicted_se,carbon_stock_Mg_ha_upper30cm,carbon_stock_Mg_ha_upper100cm,carbon_stock_Mg_ha_upper30cm_se,carbon_stock_Mg_ha_upper100cm_se,sample_ID
,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
1,14.2785,56.0953,Posidonia oceanica,0.5273098,0.08080006,0.01808088,0.06591998,0.05493332,75.37934,0.2185655,⋯,27.08706,0.0002560009,37.84513,0.01876371,0.0167864,56.29113,187.6371,50.3592,167.864,1


[1] "Carbon predicted \n"


In [4]:
print(file_result_csv)
print(file_result_txt)

[1] "/home/jovyan/Cloud Storage/naa-vre-user-data/BlueCarbon/20261104_123020-Seagrass_site_result.csv"
[1] "/home/jovyan/Cloud Storage/naa-vre-user-data/BlueCarbon/20261104_123020-Seagrass_site_result.txt"
